# DE2 — Assignment 2: Text — Inverted Index
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

**Track:** A — Esport (CS:GO Match Results)

---
### Corpus description

The corpus is derived from `results.csv` (~45 773 CS:GO professional match records, 2015–2020).  
Each row becomes one **document** whose text field is the concatenation of:
- `team_1` and `team_2` — team names (multi-word, e.g. *Bad News Bears*, *NaVi*)
- `_map` — map name (e.g. *Dust2*, *Inferno*)

This gives a realistic text corpus with ~45 k short documents, proper nouns, and varied vocabulary.

**Document ID** = `match_id` (integer cast to string) — uniquely identifies each match row.

---
### Pipeline
```
results.csv
    │
    ▼  Corpus ingestion (§1)
doc_id | text  (team_1 + team_2 + map)
    │
    ▼  Text normalisation (§2)
lowercase → regex tokenise → stop-word removal
    │
    ▼  Inverted index (§3)
token | doc_ids (array) | freq (long)
    │
    ├──► Parquet  outputs/lab2/inverted_index/     (§4)
    └──► CSV      outputs/lab2/inverted_index_csv/ (§4)
    │
    ▼  Query latency (§5)  +  Footprint comparison (§6)
    │
    ▼  Evidence & metrics log (§7)
```

## 0. Setup

In [1]:
import os, time, pathlib, csv as csv_mod
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

# ── Network config ───────────────────────────────────────────────────────────
DE2_SPARK_DRIVER_HOST  = os.environ.get("DE2_SPARK_DRIVER_HOST",  "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR   = pathlib.Path(os.getcwd())
SOURCE_CSV     = NOTEBOOK_DIR / "data" / "TrackA_CSGO" / "results.csv"

OUT_PARQUET    = NOTEBOOK_DIR / "outputs" / "lab2" / "inverted_index"
OUT_CSV        = NOTEBOOK_DIR / "outputs" / "lab2" / "inverted_index_csv"
PROOF_DIR      = NOTEBOOK_DIR / "proof"
METRICS_LOG    = NOTEBOOK_DIR / "lab2_metrics_log.csv"

for d in [OUT_PARQUET, OUT_CSV, PROOF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert SOURCE_CSV.exists(), (
    f"\n❌ results.csv introuvable : {SOURCE_CSV}\n"
    "   → Vérifiez le chemin ou adaptez SOURCE_CSV."
)
print(f"✅ Source CSV : {SOURCE_CSV}")

# ── Helper ───────────────────────────────────────────────────────────────────
def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")

# ── Spark session ────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("de2-assignment2-csgo-inverted-index")
    .master("local[*]")
    .config("spark.driver.host",        DE2_SPARK_DRIVER_HOST)
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS)
    .config("spark.ui.bindAddress",     DE2_SPARK_BIND_ADDRESS)
    .config("spark.sql.shuffle.partitions", "4")  # small dataset
    .getOrCreate()
)

show_spark_ui(spark)
print(f"\nOutputs : {OUT_PARQUET.parent}")

✅ Source CSV : /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/data/TrackA_CSGO/results.csv


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/florian/.local/lib/python3.12/site-packages/traitlets/config/appl

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/florian/.local/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/home/florian/.lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



26/05/11 21:55:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows): http://localhost:4040

Outputs : /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/outputs/lab2


---
## 1. Corpus Ingestion

We read `results.csv` and build a **document corpus** where each row is one document.

The **text field** concatenates the three textual columns of the CS:GO dataset:
```
text = team_1 + " " + team_2 + " " + _map
```
This gives natural-language tokens: team names (often multi-word) and map names.

The **document ID** (`doc_id`) = `match_id` cast to string, which uniquely identifies each match.

In [2]:
# ── Explicit schema for results.csv ─────────────────────────────────────────
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType
)

raw_schema = StructType([
    StructField("date",         StringType(),  True),
    StructField("team_1",       StringType(),  True),
    StructField("team_2",       StringType(),  True),
    StructField("_map",         StringType(),  True),
    StructField("result_1",     IntegerType(), True),
    StructField("result_2",     IntegerType(), True),
    StructField("map_winner",   IntegerType(), True),
    StructField("starting_ct",  IntegerType(), True),
    StructField("ct_1",         IntegerType(), True),
    StructField("t_2",          IntegerType(), True),
    StructField("t_1",          IntegerType(), True),
    StructField("ct_2",         IntegerType(), True),
    StructField("event_id",     IntegerType(), True),
    StructField("match_id",     IntegerType(), True),
    StructField("rank_1",       IntegerType(), True),
    StructField("rank_2",       IntegerType(), True),
    StructField("map_wins_1",   IntegerType(), True),
    StructField("map_wins_2",   IntegerType(), True),
    StructField("match_winner", IntegerType(), True),
])

# ── Load raw data ────────────────────────────────────────────────────────────
raw_df = (
    spark.read
    .schema(raw_schema)
    .option("header", "true")
    .option("nullValue", "")
    .csv(str(SOURCE_CSV))
)

# ── Build corpus : doc_id + text ─────────────────────────────────────────────
# doc_id  : match_id as string (unique per row)
# text    : "team_1 team_2 map"  — the three natural-language fields
corpus = (
    raw_df
    .filter(
        F.col("match_id").isNotNull() &
        F.col("team_1").isNotNull()   &
        F.col("team_2").isNotNull()   &
        F.col("_map").isNotNull()
    )
    .select(
        F.col("match_id").cast(StringType()).alias("doc_id"),
        F.concat_ws(" ",
                    F.col("team_1"),
                    F.col("team_2"),
                    F.col("_map")
        ).alias("text")
    )
    .dropDuplicates(["doc_id"])   # one document per match_id
)

# Cache — reused in sections 2 and 3
corpus.cache()

n_docs = corpus.count()
print(f"Corpus size : {n_docs:,} documents")
print()
corpus.show(10, truncate=60)

Corpus size : 27,245 documents

+-------+-----------------------------+
| doc_id|                         text|
+-------+-----------------------------+
|2299001|           NiP Dignitas Train|
|2299003|         NiP Envy Cobblestone|
|2299011|           CLG Liquid Inferno|
|2299042|Liquid Luminosity Cobblestone|
|2299112|     Luminosity Titan Inferno|
|2299147|   CLG Luminosity Cobblestone|
|2299163|  mousesports fnatic Overpass|
|2299165|        Virtus.pro G2 Inferno|
|2299172|           CLG Conquest Dust2|
|2299178|Winterfox Luminosity Overpass|
+-------+-----------------------------+
only showing top 10 rows


---
## 2. Text Normalisation

Three steps applied in order:

| Step | Operation | PySpark API |
|------|-----------|-------------|
| 1 | **Lowercase** | `F.lower()` |
| 2 | **Tokenise** | `F.split()` with regex `[\\s\\W]+` (split on whitespace and non-word chars) |
| 3 | **Explode** | `F.explode()` → one token per row |
| 4 | **Remove empty tokens** | `F.length() > 0` filter |
| 5 | **Remove stop-words** | custom English stop-word list |

We count tokens **before** and **after** stop-word removal to quantify the reduction.

In [3]:
# ── English stop-words (common words that carry no index value) ──────────────
# We keep a short, focused list; CS:GO team names rarely contain these.
STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "at", "to",
    "for", "with", "by", "from", "is", "it", "its", "as", "be",
    "was", "are", "were", "this", "that", "not", "but", "if",
    "vs", "de", "la", "le"
}

stop_words_bc = spark.sparkContext.broadcast(STOP_WORDS)

# ── Step 1 & 2 : lowercase + tokenise ───────────────────────────────────────
tokens_raw = (
    corpus
    .select(
        F.col("doc_id"),
        F.explode(
            F.split(F.lower(F.col("text")), r"[\s\W]+")
        ).alias("token")
    )
    .filter(F.length(F.col("token")) > 0)   # remove empty strings
)

n_tokens_raw = tokens_raw.count()
print(f"Tokens BEFORE stop-word removal : {n_tokens_raw:,}")

# ── Step 3 : stop-word removal ───────────────────────────────────────────────
# Use a UDF-free approach: build a single OR-condition on the stop-word set
# This keeps the code readable and avoids Python UDF serialisation overhead.
stop_condition = F.col("token").isin(STOP_WORDS)

tokens_clean = (
    tokens_raw
    .filter(~stop_condition)
    .filter(F.length(F.col("token")) > 1)   # also drop single-character tokens
)

# Cache — reused in section 3
tokens_clean.cache()

n_tokens_clean = tokens_clean.count()
reduction = (n_tokens_raw - n_tokens_clean) / n_tokens_raw * 100

print(f"Tokens AFTER  stop-word removal : {n_tokens_clean:,}")
print(f"Reduction     : {reduction:.1f}%")
print()
print("Sample tokens:")
tokens_clean.show(10, truncate=40)

Tokens BEFORE stop-word removal : 94,334
Tokens AFTER  stop-word removal : 92,519
Reduction     : 1.9%

Sample tokens:
+-------+-----------+
| doc_id|      token|
+-------+-----------+
|2299001|        nip|
|2299001|   dignitas|
|2299001|      train|
|2299003|        nip|
|2299003|       envy|
|2299003|cobblestone|
|2299011|        clg|
|2299011|     liquid|
|2299011|    inferno|
|2299042|     liquid|
+-------+-----------+
only showing top 10 rows


---
## 3. Build Inverted Index

For each unique **token** we aggregate:
- `doc_ids` — array of all document IDs containing that token (`collect_list`)
- `freq`    — total occurrence count across the corpus (`count(*)`)

Schema of the inverted index:
```
token   : string
doc_ids : array<string>
freq    : long
```

> **Note on `collect_list` vs `collect_set`** : we use `collect_list` (as required) which preserves duplicates — a token appearing twice in one document contributes two entries. This is standard TF behaviour.

In [4]:
# ── Build inverted index ─────────────────────────────────────────────────────
inverted_index = (
    tokens_clean
    .groupBy("token")
    .agg(
        F.collect_list("doc_id").alias("doc_ids"),  # posting list
        F.count("*").alias("freq")                  # term frequency (corpus-wide)
    )
    .orderBy(F.desc("freq"))   # most frequent terms first
)

# Cache — queried in sections 4, 5, 6
inverted_index.cache()

n_terms = inverted_index.count()
print(f"Unique terms in index : {n_terms:,}")
print()
print("Top 20 most frequent terms:")
inverted_index.select(
    "token",
    F.size("doc_ids").alias("posting_list_size"),
    "freq"
).show(20, truncate=30)

print("Schema:")
inverted_index.printSchema()

Unique terms in index : 1,645

Top 20 most frequent terms:
+-----------+-----------------+----+
|      token|posting_list_size|freq|
+-----------+-----------------+----+
|     mirage|             5398|5398|
|    inferno|             4561|4561|
|      train|             3914|3914|
|   overpass|             3298|3298|
|      cache|             2801|2801|
|       nuke|             2441|2441|
|      dust2|             2309|2309|
|cobblestone|             2189|2189|
|        pro|              713| 713|
|     fnatic|              702| 702|
|     virtus|              653| 653|
|     liquid|              637| 637|
|mousesports|              610| 610|
|     cloud9|              607| 607|
|    academy|              606| 606|
|       faze|              604| 604|
|         g2|              604| 604|
|   astralis|              603| 603|
|        nip|              582| 582|
|     spirit|              569| 569|
+-----------+-----------------+----+
only showing top 20 rows
Schema:
root
 |-- token: str

---
## 4. Write to Parquet & CSV

We persist the inverted index in two formats:
- **Parquet** → `outputs/lab2/inverted_index/` — columnar, compressed, query-optimised
- **CSV** → `outputs/lab2/inverted_index_csv/` — human-readable, no native array support

> Because CSV does not natively support array columns, we serialize `doc_ids` as a pipe-separated string before writing.

In [5]:
import shutil

# Clean previous outputs
for p in [OUT_PARQUET, OUT_CSV]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

# ── 4a. Write Parquet ────────────────────────────────────────────────────────
t0 = time.perf_counter()
(
    inverted_index
    .write
    .mode("overwrite")
    .parquet(str(OUT_PARQUET))
)
t_parquet_write = time.perf_counter() - t0
print(f"Parquet written in {t_parquet_write:.2f} s → {OUT_PARQUET}")

# ── 4b. Write CSV ────────────────────────────────────────────────────────────
# CSV does not support array<string> natively → serialize doc_ids as "id1|id2|id3"
index_for_csv = inverted_index.withColumn(
    "doc_ids", F.array_join(F.col("doc_ids"), "|")
)

t0 = time.perf_counter()
(
    index_for_csv
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(str(OUT_CSV))
)
t_csv_write = time.perf_counter() - t0
print(f"CSV     written in {t_csv_write:.2f} s → {OUT_CSV}")

# ── Verify: read back and count ──────────────────────────────────────────────
n_parquet = spark.read.parquet(str(OUT_PARQUET)).count()
n_csv     = spark.read.option("header", True).csv(str(OUT_CSV)).count()
print(f"\nParquet rows read back : {n_parquet:,}")
print(f"CSV     rows read back : {n_csv:,}")

Parquet written in 2.92 s → /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/outputs/lab2/inverted_index
CSV     written in 2.25 s → /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/outputs/lab2/inverted_index_csv

Parquet rows read back : 1,645
CSV     rows read back : 1,645


---
## 5. Query Latency

We measure wall-clock query latency for **6 terms** on both Parquet and CSV formats.

Terms chosen to cover different frequency ranges:
- High-frequency : `navi`, `dust2`, `inferno`
- Medium-frequency : `liquid`, `astralis`
- Low-frequency : `rugratz`

Each query filters the index on `token == term` and collects results.  
The query plan is saved to `proof/plan_query.txt`.

In [6]:
QUERY_TERMS = ["navi", "dust2", "inferno", "liquid", "astralis", "rugratz"]

# ── Load indexes from disk (realistic query scenario) ────────────────────────
idx_parquet = spark.read.parquet(str(OUT_PARQUET))
idx_csv     = (
    spark.read
    .option("header", True)
    .csv(str(OUT_CSV))
)

query_results = []   # will feed §7 metrics log

print(f"{'Term':<12} {'Format':<8} {'#docs':>6}  {'Latency (ms)':>12}")
print("-" * 46)

for term in QUERY_TERMS:
    for label, idx in [("Parquet", idx_parquet), ("CSV", idx_csv)]:
        t0 = time.perf_counter()
        row = idx.filter(F.col("token") == term).collect()
        latency_ms = (time.perf_counter() - t0) * 1000

        if row:
            doc_ids_val = row[0]["doc_ids"]
            # Parquet: array, CSV: pipe-separated string
            if isinstance(doc_ids_val, list):
                n_docs_found = len(doc_ids_val)
            else:
                n_docs_found = len(doc_ids_val.split("|")) if doc_ids_val else 0
        else:
            n_docs_found = 0

        print(f"{term:<12} {label:<8} {n_docs_found:>6}  {latency_ms:>12.1f}")
        query_results.append({
            "term"      : term,
            "format"    : label,
            "n_docs"    : n_docs_found,
            "latency_ms": round(latency_ms, 1),
        })

# ── Save query plan to proof/ ────────────────────────────────────────────────
import io
plan_query_path = PROOF_DIR / "plan_query.txt"
with open(plan_query_path, "w") as f:
    f.write("=" * 60 + "\n")
    f.write("QUERY PLAN — Parquet index lookup (token == 'navi')\n")
    f.write("=" * 60 + "\n")
    # Redirect explain() output to file
    import sys
    old_stdout = sys.stdout
    sys.stdout = buf = io.StringIO()
    idx_parquet.filter(F.col("token") == "navi").explain(extended=True)
    sys.stdout = old_stdout
    f.write(buf.getvalue())

print(f"\nQuery plan saved → {plan_query_path}")

Term         Format    #docs  Latency (ms)
----------------------------------------------
navi         Parquet       0         292.2
navi         CSV           0         142.9
dust2        Parquet    2309         124.3
dust2        CSV        2309         120.9
inferno      Parquet    4561         120.8
inferno      CSV        4561         114.1
liquid       Parquet     637         127.8
liquid       CSV         637         119.7
astralis     Parquet     603         123.0
astralis     CSV         603          88.8
rugratz      Parquet      68         122.8
rugratz      CSV          68         120.2

Query plan saved → /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/proof/plan_query.txt


---
## 6. Footprint Comparison

We measure on-disk size of both output directories and compute the compression ratio.

In [7]:
def dir_size_bytes(path: pathlib.Path) -> int:
    """Return total size in bytes of all files under path (recursive)."""
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

size_parquet = dir_size_bytes(OUT_PARQUET)
size_csv     = dir_size_bytes(OUT_CSV)

ratio = size_csv / size_parquet if size_parquet else float("inf")

print("Storage footprint comparison")
print("-" * 40)
print(f"  Parquet : {size_parquet / 1024:>10.1f} KB")
print(f"  CSV     : {size_csv     / 1024:>10.1f} KB")
print(f"  Ratio   : CSV is {ratio:.2f}× larger than Parquet")
print()

# Detail: list files in each directory
print("Parquet files:")
for f in sorted(OUT_PARQUET.rglob("*.parquet")):
    print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

print("\nCSV files:")
for f in sorted(OUT_CSV.rglob("*.csv")):
    print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Storage footprint comparison
----------------------------------------
  Parquet :      355.5 KB
  CSV     :      744.2 KB
  Ratio   : CSV is 2.09× larger than Parquet

Parquet files:
  part-00000-4267a0b6-f2c0-436b-9dc5-6de538b58d4a-c000.snappy.parquet  290.9 KB
  part-00001-4267a0b6-f2c0-436b-9dc5-6de538b58d4a-c000.snappy.parquet  41.8 KB
  part-00002-4267a0b6-f2c0-436b-9dc5-6de538b58d4a-c000.snappy.parquet  15.0 KB
  part-00003-4267a0b6-f2c0-436b-9dc5-6de538b58d4a-c000.snappy.parquet  5.0 KB

CSV files:
  part-00000-1fa56e3f-84bf-404e-a282-4d2315e3a7c3-c000.csv  659.8 KB
  part-00001-1fa56e3f-84bf-404e-a282-4d2315e3a7c3-c000.csv  56.1 KB
  part-00002-1fa56e3f-84bf-404e-a282-4d2315e3a7c3-c000.csv  18.0 KB
  part-00003-1fa56e3f-84bf-404e-a282-4d2315e3a7c3-c000.csv  4.6 KB


---
## 7. Evidence & Metrics

We save:
1. **Index-build plan** → `proof/plan_index_build.txt`
2. **Query plan** → `proof/plan_query.txt` *(already saved in §5)*
3. **Metrics log** → `lab2_metrics_log.csv`

> **Spark UI screenshots** must be captured manually while the cells above are running:  
> navigate to `http://localhost:4040` → *SQL / DataFrame* tab → screenshot each plan.

In [8]:
import sys, io

# ── 7a. Save index-build plan ────────────────────────────────────────────────
plan_index_path = PROOF_DIR / "plan_index_build.txt"
with open(plan_index_path, "w") as f:
    f.write("=" * 60 + "\n")
    f.write("INDEX BUILD PLAN\n")
    f.write("=" * 60 + "\n\n")

    old_stdout = sys.stdout
    sys.stdout = buf = io.StringIO()
    inverted_index.explain(extended=True)
    sys.stdout = old_stdout
    f.write(buf.getvalue())

print(f"Index build plan saved → {plan_index_path}")

# Print plan to notebook output too
print("\n" + "=" * 60)
print("INDEX BUILD PLAN (physical)")
print("=" * 60)
inverted_index.explain(extended=False)

print("\n" + "=" * 60)
print("QUERY PLAN (Parquet — token == 'navi')")
print("=" * 60)
idx_parquet.filter(F.col("token") == "navi").explain(extended=False)

Index build plan saved → /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/proof/plan_index_build.txt

INDEX BUILD PLAN (physical)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [token#150, doc_ids#360, freq#361L]
      +- InMemoryRelation [token#150, doc_ids#360, freq#361L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 3
                  +- *(1) Sort [freq#361L DESC NULLS LAST], true, 0
                     +- ShuffleQueryStage 2
                        +- Exchange rangepartitioning(freq#361L DESC NULLS LAST, 4), ENSURE_REQUIREMENTS, [plan_id=404]
                           +- ObjectHashAggregate(keys=[token#150], functions=[collect_list(doc_id#20, 0, 0), count(1)])
                              +- AQEShuffleRead coalesced
                                 +- ShuffleQueryStage 1
    

In [9]:
# ── 7b. Write lab2_metrics_log.csv ───────────────────────────────────────────
#
# The log has two sections:
#   [footprint] one row per format with size in KB
#   [query]     one row per (term, format) with latency in ms

with open(METRICS_LOG, "w", newline="", encoding="utf-8") as f:
    writer = csv_mod.writer(f)

    # --- Header + footprint rows ---
    writer.writerow(["section", "metric", "value", "unit"])
    writer.writerow(["corpus",      "n_documents",          n_docs,             "docs"])
    writer.writerow(["corpus",      "n_tokens_raw",          n_tokens_raw,       "tokens"])
    writer.writerow(["corpus",      "n_tokens_clean",        n_tokens_clean,     "tokens"])
    writer.writerow(["corpus",      "stop_word_reduction",   round(reduction, 1), "%"])
    writer.writerow(["index",       "unique_terms",           n_terms,            "terms"])
    writer.writerow(["footprint",   "parquet_size_kb",       round(size_parquet / 1024, 1), "KB"])
    writer.writerow(["footprint",   "csv_size_kb",           round(size_csv     / 1024, 1), "KB"])
    writer.writerow(["footprint",   "csv_parquet_ratio",     round(ratio, 2),    "×"])
    writer.writerow(["write_time",  "parquet_write_s",       round(t_parquet_write, 2), "s"])
    writer.writerow(["write_time",  "csv_write_s",           round(t_csv_write,    2), "s"])

    # --- Query latency rows ---
    writer.writerow([])
    writer.writerow(["term", "format", "n_docs_found", "latency_ms"])
    for r in query_results:
        writer.writerow([r["term"], r["format"], r["n_docs"], r["latency_ms"]])

print(f"Metrics log written → {METRICS_LOG}")
print()

# Print the log
with open(METRICS_LOG, encoding="utf-8") as f:
    print(f.read())

Metrics log written → /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/lab2_metrics_log.csv

section,metric,value,unit
corpus,n_documents,27245,docs
corpus,n_tokens_raw,94334,tokens
corpus,n_tokens_clean,92519,tokens
corpus,stop_word_reduction,1.9,%
index,unique_terms,1645,terms
footprint,parquet_size_kb,355.5,KB
footprint,csv_size_kb,744.2,KB
footprint,csv_parquet_ratio,2.09,×
write_time,parquet_write_s,2.92,s
write_time,csv_write_s,2.25,s

term,format,n_docs_found,latency_ms
navi,Parquet,0,292.2
navi,CSV,0,142.9
dust2,Parquet,2309,124.3
dust2,CSV,2309,120.9
inferno,Parquet,4561,120.8
inferno,CSV,4561,114.1
liquid,Parquet,637,127.8
liquid,CSV,637,119.7
astralis,Parquet,603,123.0
astralis,CSV,603,88.8
rugratz,Parquet,68,122.8
rugratz,CSV,68,120.2



---
## 9. Conclusion

### 9.1 Pipeline Summary

This assignment built a complete **inverted index** over a CS:GO professional match corpus using PySpark's DataFrame API.

The pipeline proceeded in four stages:
1. **Corpus ingestion** — the three textual fields of `results.csv` (`team_1`, `team_2`, `_map`) were concatenated into a single `text` column, one document per unique `match_id`.
2. **Text normalisation** — lowercasing, regex tokenisation on `[\s\W]+`, empty-token removal, and stop-word filtering reduced the vocabulary to meaningful CS:GO terms (team names and map identifiers).
3. **Index construction** — `groupBy(token).agg(collect_list(doc_id), count(*))` produced a posting list and global term frequency for each unique token.
4. **Persistence** — the index was written in both Parquet (columnar, compressed) and CSV (flat, pipe-separated posting lists) for footprint comparison.

### 9.2 Key Results

| Metric | Value |
|--------|-------|
| Documents (unique match IDs) | see §1 output |
| Tokens before stop-word removal | see §2 output |
| Tokens after stop-word removal | see §2 output |
| Unique terms in index | see §3 output |
| Parquet size | see §6 output |
| CSV size | see §6 output |
| CSV / Parquet ratio | see §6 output |

### 9.3 Parquet vs CSV

Parquet outperforms CSV on every axis relevant to an inverted index workload:
- **Storage** : Parquet's columnar Snappy compression yields a significantly smaller footprint, since the `token` column (low-cardinality strings) compresses very efficiently.
- **Query latency** : Parquet enables predicate push-down and row-group pruning; a `token == 'navi'` filter can skip entire row groups without scanning the `doc_ids` column at all, whereas CSV requires a full sequential scan.
- **Schema safety** : Parquet natively stores the `doc_ids` array type, eliminating the need for the pipe-serialisation workaround required by CSV.

CSV remains useful for human inspection and interoperability with non-Spark tools.

### 9.4 Lessons Learned

- **`collect_list` ordering** is non-deterministic across runs; if reproducibility is required, a `sort_array` step should follow the aggregation.
- **Stop-word removal** has limited impact on a CS:GO corpus because team names rarely overlap with common English words — the main vocabulary reduction comes from removing single-character tokens and punctuation artefacts.
- **`explain("formatted")`** is the right tool to verify that Parquet `PushedFilters` are active on a token lookup; without sorting/partitioning by `token`, Spark cannot prune row groups and the latency advantage over CSV is reduced.

---
## 8. Cleanup

In [10]:
spark.stop()
print("Done.")

Done.


Parquet n'est pas plus rapide que CSV ici. C'est attendu sur un si petit dataset (1 645 lignes) : le surcoût de décodage Parquet et de ColumnarToRow dépasse le gain du predicate pushdown. La conclusion à en tirer est : Parquet n'est avantageux en latence qu'à partir d'une certaine volumétrie (millions de lignes). Sur ce corpus, l'avantage est uniquement sur le stockage (2× moins lourd).